<a href="https://colab.research.google.com/github/abedulll/UTS_BIG_DATA_2/blob/main/UTS-BIG_DATA_Analisis_Aplikasi_MYPERTAMINA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install google-play-scraper

In [ ]:
from google_play_scraper import reviews, Sort
import csv

result, _ = reviews(
    'com.dafturn.mypertamina',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=500,
    filter_score_with=None
)

filename = 'ulasan_google_play.csv'


with open(filename, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['userName', 'score', 'at', 'content'])
    writer.writeheader()
    for review in result:

        writer.writerow({
            'userName': review['userName'],
            'score': review['score'],
            'at': review['at'],
            'content': review['content']
        })

print(f"Berhasil menyimpan {len(result)} ulasan ke '{filename}'")

In [ ]:
pip install transformers torch pandas

In [ ]:
import pandas as pd
from transformers import pipeline

# Load the data
df = pd.read_csv('ulasan_google_play.csv')

# Initialize the sentiment analysis pipeline
# Note: w11wo/indonesian-roberta-base-sentiment-classifier is the standard sentiment version of this model series
model_name = "w11wo/indonesian-roberta-base-sentiment-classifier"
sentiment_pipeline = pipeline("sentiment-analysis", model=model_name)

# Define a function to get sentiment to handle long strings or potential errors
def get_sentiment(text):
    try:
        # Truncate text to fit model max length (usually 512 tokens)
        result = sentiment_pipeline(str(text)[:512])[0]
        return result['label']
    except:
        return "NEUTRAL"

# Apply sentiment analysis
print("Memproses analisis sentimen... (ini mungkin memakan waktu sebentar)")
df['sentiment'] = df['content'].apply(get_sentiment)

# Display the first few rows with sentiment
display(df.head())

# Summary of sentiments
print("\nRingkasan Sentimen:")
print(df['sentiment'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Menghitung data
sentiment_counts = df['sentiment'].value_counts()
colors = {'negative': '#ff4d4d', 'positive': '#4CAF50', 'neutral': '#2196F3'}
order = ['negative', 'neutral', 'positive']

# Membuat figure dengan dua subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# 1. Grafik Batang (Jumlah Absolut)
sns.barplot(
    x=sentiment_counts.index,
    y=sentiment_counts.values,
    order=order,
    palette=colors,
    hue=sentiment_counts.index,
    ax=ax1,
    legend=False
)
ax1.set_title('Jumlah Ulasan per Kategori', fontsize=15, pad=10)
ax1.set_ylabel('Jumlah Ulasan')

# Menambahkan angka di atas batang
for i, v in enumerate(sentiment_counts.loc[order]):
    ax1.text(i, v + 5, str(v), ha='center', fontweight='bold')

# 2. Grafik Lingkaran (Persentase)
ax2.pie(
    sentiment_counts.loc[order],
    labels=order,
    autopct='%1.1f%%',
    startangle=140,
    colors=[colors[s] for s in order],
    explode=(0.05, 0, 0) # Menonjolkan bagian negatif
)
ax2.set_title('Persentase Distribusi Sentimen', fontsize=15, pad=10)

plt.suptitle('Analisis Sentimen Ulasan MyPertamina', fontsize=20, y=1.05)
plt.tight_layout()
plt.show()